### ライブラリの読み込み

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from collections import defaultdict

### データの読み込み

In [2]:
# TCVファイルの読み込み
train_data = pd.read_csv('../000.data/train/train_C.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_C" のテストデータだけを抽出
test_data = test[test["user_id"].str.endswith("_C")]

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

### 特徴量エンジニアリング

In [5]:
# タイムスタンプをdatetime型に変換
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])

In [6]:
# 最新の行動からの経過時間（秒）
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [7]:
# ユーザーごとの行動回数集計
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean")
).reset_index()

In [8]:
# 商品ごとの人気度
product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique")
).reset_index()

In [9]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")

In [10]:
# 特徴量を追加（広告クリックや購入に特化した特徴量）
train_data["ad_purchase_event"] = train_data["event_type"].apply(lambda x: 1 if x == 3 else 0)  # 購入イベント
train_data["ad_click_event"] = train_data["event_type"].apply(lambda x: 1 if x == 2 else 0)  # 広告クリック

### XGBoost 用データセットの準備

In [11]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "product_total_interactions", "product_unique_users", 
    "ad_purchase_event", "ad_click_event"
]

In [12]:
# クラス不均衡に対する学習データのバランス調整
# 購入イベントと広告クリックのデータが少ないので、少数派クラスをオーバーサンプリング
train_data_pos = train_data[train_data["relevance"] >= 2]  # 広告クリックと購入のデータ
train_data_neg = train_data[train_data["relevance"] < 2]   # その他のデータ

In [13]:
# 少数派クラス（購入・広告クリック）をオーバーサンプリング
train_data_pos_oversampled = resample(train_data_pos, 
                                      replace=True, 
                                      n_samples=len(train_data_neg),  # 多数派クラスのサンプル数に合わせる
                                      random_state=42)

In [14]:
# サンプリング後のデータ
train_data_balanced = pd.concat([train_data_neg, train_data_pos_oversampled])

In [15]:
# ユーザー単位での分割
unique_users = train_data_balanced["user_id"].unique()
train_users, val_users = train_test_split(unique_users, test_size=0.2, random_state=42)

In [16]:
# 訓練データとバリデーションデータをフィルタリング
train_set = train_data_balanced[train_data_balanced["user_id"].isin(train_users)]
val_set = train_data_balanced[train_data_balanced["user_id"].isin(val_users)]

In [17]:
# 特徴量とラベル
X_train = train_set[features]
y_train = train_set["relevance"]
X_val = val_set[features]
y_val = val_set["relevance"]

In [18]:
# クエリごとのデータ数（ユーザーごと）
group_train = train_set.groupby("user_id").size().to_numpy()
group_val = val_set.groupby("user_id").size().to_numpy()

In [19]:
# XGBoost DMatrix に変換
dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_group(group_train)

dval = xgb.DMatrix(X_val, label=y_val)
dval.set_group(group_val)

### モデルの学習

In [20]:
# XGBoostのパラメータ設定
params = {
    "objective": "rank:ndcg",
    "eval_metric": "ndcg",
    "booster": "gbtree",
    "eta": 0.1,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "gamma": 0.1,
    "max_delta_step": 1  # 不均衡なデータに対して調整
}

In [21]:
# 学習
model = xgb.train(
    params, dtrain, num_boost_round=200,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=20, verbose_eval=10
)

[0]	train-ndcg:0.99985	val-ndcg:0.99984
[10]	train-ndcg:0.99969	val-ndcg:0.99960
[20]	train-ndcg:0.99983	val-ndcg:0.99983
[30]	train-ndcg:0.99991	val-ndcg:0.99988
[40]	train-ndcg:0.99998	val-ndcg:1.00000
[50]	train-ndcg:0.99998	val-ndcg:0.99999
[60]	train-ndcg:0.99998	val-ndcg:0.99999
[62]	train-ndcg:0.99998	val-ndcg:0.99999


### nDCG の計算関数

In [22]:
def dcg_at_k(relevance_scores, k):
    relevance_scores = np.asarray(relevance_scores)[:k]
    return np.sum(relevance_scores / np.log2(np.arange(2, relevance_scores.size + 2)))

In [23]:
def ndcg_at_k(relevance_scores, k):
    dcg = dcg_at_k(relevance_scores, k)
    ideal_dcg = dcg_at_k(sorted(relevance_scores, reverse=True), k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0

In [24]:
def evaluate_ndcg(data, ground_truth, k=22):
    ndcg_scores = []
    for user_id, group in data.groupby("user_id"):
        predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
        relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
        ndcg_scores.append(ndcg_at_k(relevance_scores, k))
    return np.mean(ndcg_scores) if ndcg_scores else 0

In [25]:
# バリデーションデータの nDCG 計算
ground_truth_val = defaultdict(dict)
for _, row in val_set.iterrows():
    ground_truth_val[row["user_id"]][row["product_id"]] = row["relevance"]

# バリデーションデータのコピーを作成
val_set_copy = val_set.copy()

# バリデーションデータに対してスコアを設定
val_set_copy["score"] = model.predict(xgb.DMatrix(val_set_copy[features]))

# nDCG の計算
ndcg_val = evaluate_ndcg(val_set_copy, ground_truth_val, k=22)
print(f"Validation nDCG@22: {ndcg_val:.4f}")

Validation nDCG@22: 0.9356


### 予測と提出データの作成

In [26]:
# 推薦候補商品リストを作成
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [27]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [28]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
# train_data における各ユーザーごとの広告クリックイベントと購入イベントの集計
user_event_counts = train_data.groupby('user_id')['event_type'].value_counts().unstack(fill_value=0)

In [29]:
# 'event_type' の値に応じて広告クリックと購入を定義
user_event_counts['ad_click_event'] = user_event_counts.get(2, 0)  # event_type == 2 は広告クリック
user_event_counts['ad_purchase_event'] = user_event_counts.get(3, 0)  # event_type == 3 は購入

In [30]:
# test_data にマージ
test_data = test_data.merge(user_event_counts[['ad_click_event', 'ad_purchase_event']], on='user_id', how='left')

In [31]:
# NaN 値が存在する場合、0で埋める
test_data.fillna({'ad_click_event': 0, 'ad_purchase_event': 0}, inplace=True)

In [32]:
# `test_data` に `user_features` や `product_features` をマージ（必要に応じて）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")

In [33]:
# 欠損値処理（初めてのユーザーや商品に対して）
test_data.fillna(0, inplace=True)

In [34]:
# 予測用データ
X_test = test_data[features]
dtest = xgb.DMatrix(X_test)

In [35]:
# 予測スコアの計算
test_data["score"] = model.predict(dtest)

In [36]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [37]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.0396


In [38]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]

In [39]:
# rankを整数に変換
submission['rank'] = submission['rank'].astype(int)

In [40]:
# 保存
submission.to_csv('./predict_file/predictions_C.tsv', sep="\t", index=False)